# grad-accumulate-on-leaf — ex2: zero_grad(params): clear leaf.grad to None across an iterable of params

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-accumulate-on-leaf`. Running the final beacon cell reports progress against the `Backprop: Grad accumulate on leaf` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad accumulate on leaf` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-accumulate-on-leaf`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-accumulate-on-leaf"
DD_SUBTOPIC = "Backprop: Grad accumulate on leaf"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad accumulate → zero_grad cycle — quick refresher

Ex1's `accumulate_grad` ALWAYS adds (never overwrites). This is the right behavior for a single backward pass through a graph with shared parameters. But across training STEPS, last step's gradient must be cleared — otherwise it leaks into the next step's update.

Canonical training loop:
```python
for batch in loader:
    optimizer.zero_grad()           # 1. clear last step's grads
    loss = forward(batch)
    loss.backward()                  # 2. accumulate this step's grads
    optimizer.step()                 # 3. apply update
```

`zero_grad` is the canonical way to clear: for each Parameter, set `.grad = None` (or zero it). PyTorch's `set_to_none=True` (default in 2.0+) is `.grad = None` — cheaper than zeroing because it skips an allocation.

### Exercise 2 — zero_grad(params): clear leaf.grad to None across an iterable of params

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the zero_grad pattern at the training-step boundary: walk an iterable of leaves and set each `.grad = None`, allowing the next step's accumulate to start fresh.
> Keywords: zero-grad, training-loop, leaf, set-to-none, step-boundary
> ```

**KCs targeted:** `grad-accumulate-on-leaf`, `parameter-subclass-of-tensor`

Implement two functions, building on ex1:

**1. `accumulate_grad(leaf, g)`** — same as ex1. Set `leaf.grad = g` on first touch (when `.grad is None`), else `leaf.grad = leaf.grad + g`. (Use rebinding `+`, not `+=`.)

**2. `zero_grad(params)`** — given an iterable of MiniTensors, set each one's `.grad = None`. This is the `set_to_none=True` PyTorch convention (default in PyTorch 2.0+):

```python
def zero_grad(params):
    for p in params:
        p.grad = None
```

Rules:
- `None` (not `t.zeros_like(p.array)`). Setting to None means the next `accumulate_grad` takes the first-touch (rebind) path — skipping an allocation.
- Accept any iterable, not just a list — generators, tuples, `module.parameters()` all should work.

**The round-trip invariant** is the load-bearing test for ex2:

```
step 1:  accumulate(p, g1); accumulate(p, g2)  → p.grad = g1+g2
         zero_grad([p])                         → p.grad = None
step 2:  accumulate(p, g3); accumulate(p, g4)  → p.grad = g3+g4   (NOT g1+g2+g3+g4!)
```

Without `zero_grad`, the second step's `.grad` would be `g1+g2+g3+g4` — last step's gradients corrupting this step's update. That's the bug `zero_grad` exists to prevent.

In [ ]:
def accumulate_grad(leaf: MiniTensor, g) -> None:
    """Set leaf.grad on first touch, add on subsequent."""
    raise NotImplementedError()


def zero_grad(params) -> None:
    """Set leaf.grad = None for each leaf in params (set-to-none convention)."""
    raise NotImplementedError()


def _test_ex2():
    # --- accumulate_grad still behaves per ex1 ---
    p = MiniTensor(t.zeros(3), requires_grad=True)
    accumulate_grad(p, t.tensor([1.0, 2.0, 3.0]))
    accumulate_grad(p, t.tensor([10.0, 20.0, 30.0]))
    assert t.allclose(p.grad, t.tensor([11.0, 22.0, 33.0]))

    # --- zero_grad: single leaf, .grad → None ---
    zero_grad([p])
    assert p.grad is None, (
        f'zero_grad must set .grad to None (set-to-none convention), got {p.grad}'
    )

    # --- zero_grad accepts an iterable (generator), not just a list ---
    p2 = MiniTensor(t.zeros(2), requires_grad=True)
    p3 = MiniTensor(t.zeros(2), requires_grad=True)
    accumulate_grad(p2, t.ones(2))
    accumulate_grad(p3, t.ones(2))
    zero_grad(iter([p2, p3]))   # generator/iterator, not a list
    assert p2.grad is None and p3.grad is None

    # --- zero_grad handles already-None grad (no crash) ---
    p4 = MiniTensor(t.zeros(2), requires_grad=True)
    assert p4.grad is None
    zero_grad([p4])  # idempotent
    assert p4.grad is None

    # --- THE LOAD-BEARING TEST: round-trip invariance across steps ---
    # step 1: accumulate g1, g2
    # zero_grad
    # step 2: accumulate g3, g4
    # Expected: after step 2, p.grad == g3 + g4 (NOT g1+g2+g3+g4)
    p = MiniTensor(t.zeros(3), requires_grad=True)
    g1, g2, g3, g4 = (
        t.tensor([1.0, 0.0, 0.0]),
        t.tensor([0.0, 1.0, 0.0]),
        t.tensor([0.0, 0.0, 1.0]),
        t.tensor([1.0, 1.0, 1.0]),
    )
    # step 1
    accumulate_grad(p, g1)
    accumulate_grad(p, g2)
    assert t.allclose(p.grad, g1 + g2)

    zero_grad([p])

    # step 2 — fresh accumulation, last step's gradient must be gone
    accumulate_grad(p, g3)
    accumulate_grad(p, g4)
    expected = g3 + g4
    leaked = g1 + g2 + g3 + g4  # what we'd see WITHOUT zero_grad
    assert t.allclose(p.grad, expected), (
        f'zero_grad failed: p.grad should be {expected} (step-2 only), got {p.grad}'
    )
    assert not t.allclose(p.grad, leaked), (
        'p.grad still contains step-1 contributions — zero_grad did not clear'
    )

    # --- after zero_grad, the FIRST accumulate post-zero takes the first-touch path ---
    # (the .grad rebinds to the input tensor exactly, no addition)
    p = MiniTensor(t.zeros(2), requires_grad=True)
    accumulate_grad(p, t.tensor([5.0, 6.0]))
    zero_grad([p])
    fresh_g = t.tensor([100.0, 200.0])
    accumulate_grad(p, fresh_g)
    assert p.grad is fresh_g, (
        'first-post-zero accumulate must take first-touch path (rebind, not add) — '
        'this is why set-to-none beats set-to-zeros (skips an allocation + add)'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def accumulate_grad(leaf: MiniTensor, g) -> None:
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g


def zero_grad(params) -> None:
    for p in params:
        p.grad = None
```

**Why `None`, not zeros.** PyTorch 2.0 switched `zero_grad`'s default to `set_to_none=True` because:
(a) zero allocation cost vs allocating `t.zeros_like(p.grad)`,
(b) the FIRST accumulate after a `None` reset takes the rebind path (just stores the incoming tensor) instead of `zeros + g` — saves an addition,
(c) downstream code that does `if p.grad is None: ...` works as expected.

**Why a separate function from accumulate_grad.** The split is the natural division of responsibility: `accumulate_grad` runs during a single backward pass (called by every leaf-touching back-fn), `zero_grad` runs once per training step (called from the training loop). Coupling them would make multi-step accumulation (e.g. gradient accumulation across micro-batches) awkward.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()